# Tratamento dos Dados — Casos por ano de diagóstico, faixa etária, escolaridade e raça cor


---
## 1. Imports

In [16]:
from __future__ import annotations

import re
import unicodedata
from io import StringIO
from pathlib import Path

import pandas as pd

---
## 2. Configurações Gerais


In [17]:
# ── Caminhos ─────────────────────────────────────────────────────────────────
BASE_VARIAVEIS = Path("/home/anna/tcc/Variáveis/Casos")
OUT_DIR        = Path("/home/anna/tcc/Seleção e Transformação/Casos")

# ── Subpastas de cada dimensão ────────────────────────────────────────────────
DIR_ANO_DIAG    = BASE_VARIAVEIS / "Ano Diagnóstico" / "Dados"
DIR_ESCOLARIDADE = BASE_VARIAVEIS / "Escolaridade"   / "Dados"
DIR_FAIXA_ETARIA = BASE_VARIAVEIS / "Faixa Etária"   / "Dados"
DIR_RACA_COR     = BASE_VARIAVEIS / "Raça-Cor"        / "Dados"

# ── Nomes dos arquivos de saída ───────────────────────────────────────────────
SAIDA_ANO_DIAG    = OUT_DIR / "incidencia _ano_diagnostico.csv"
SAIDA_ESCOLARIDADE = OUT_DIR / "incidencia _escolaridade.csv"
SAIDA_FAIXA_ETARIA = OUT_DIR / "incidencia _faixa_etaria.csv"
SAIDA_RACA_COR     = OUT_DIR / "incidencia _raca_cor.csv"
SAIDA_TABELA_UNIDA = OUT_DIR / "tabela_incidencia _.csv"

# ── Faixas etárias padronizadas (destino após agrupamento) ────────────────────
FAIXA_TARGET = [
    "0-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70 ou mais",   # unifica 70-79 e 80 ou mais
]

# ── Intervalo válido de anos de diagnóstico ───────────────────────────────────
ANO_MIN_VALIDO = 1985
ANO_MAX_VALIDO = 2023

OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 3. Funções de Normalização de Texto

Funções utilitárias para padronizar texto, nomes de colunas e extrair informações dos campos brutos.  
Todas as demais etapas dependem dessas funções — por isso ficam definidas primeiro.

In [18]:
def normalize_text(value: str) -> str:
    """Remove acentos, converte para minúsculas e colapsa espaços múltiplos."""
    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r"\s+", " ", text)
    return text


def clean_col(col: str) -> str:
    """Remove BOM, aspas e normaliza o nome de coluna sem alterar maiúsculas."""
    col = str(col).replace("\ufeff", "").replace('"', "").strip()
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    col = re.sub(r"\s+", " ", col).strip()
    return col


def parse_municipio_codigo(procedencia: str) -> tuple[str | None, int | None]:
    """
    Extrai município e código IBGE do campo de procedência.
    Formato esperado: 'MG - 3100000 - Nome do Município'
    Retorna (municipio_normalizado, codigo_ibge) ou (None, None) se não casar.
    """
    text  = str(procedencia).replace('"', "").strip()
    match = re.match(r"^MG\s*-\s*(\d{7})\s*-\s*(.+)$", text, flags=re.IGNORECASE)
    if not match:
        return None, None
    codigo    = int(match.group(1))
    municipio = normalize_text(match.group(2))
    return municipio, codigo


def extract_year(file_name: str) -> int:
    """Extrai o ano (19xx ou 20xx) do nome do arquivo. Lança ValueError se não encontrar."""
    match = re.search(r"(19\d{2}|20\d{2})", file_name)
    if not match:
        raise ValueError(f"Ano não encontrado no nome do arquivo: {file_name}")
    return int(match.group(1))


---
## 4. Funções de Leitura dos Arquivos Brutos

Os arquivos brutos do RHC/INCA chegam como CSVs com encoding `latin1`, separador `;` e
linhas de cabeçalho não padronizadas. A função `read_raw_table` localiza automaticamente
a linha de cabeçalho real (que contém a coluna de procedência) e descarta as linhas anteriores.

In [19]:
def read_raw_table(file_path: Path) -> pd.DataFrame:
    """
    Lê um arquivo bruto do RHC/INCA (latin1, sep=';') localizando o cabeçalho real.
    Adiciona as colunas MUNICIPIOS e CODIGO a partir do campo de procedência.
    Descarta linhas sem código IBGE válido.
    """
    content    = file_path.read_text(encoding="latin1", errors="ignore").splitlines()
    header_idx = None
    for idx, line in enumerate(content):
        if ";" in line and "proced" in normalize_text(line):
            header_idx = idx
            break
    if header_idx is None:
        raise ValueError(f"Cabeçalho não encontrado em: {file_path}")

    table_text = "\n".join(content[header_idx:])
    df = pd.read_csv(StringIO(table_text), sep=";", engine="python", encoding="latin1")
    df.columns = [clean_col(c) for c in df.columns]

    # Remove colunas sem nome
    unnamed = [c for c in df.columns if clean_col(c) == ""]
    if unnamed:
        df = df.drop(columns=unnamed)

    # Extrai município e código da primeira coluna (procedência)
    col_proc = df.columns[0]
    parsed   = df[col_proc].apply(parse_municipio_codigo)
    df["MUNICIPIOS"] = parsed.map(lambda x: x[0])
    df["CODIGO"]     = parsed.map(lambda x: x[1])

    # Mantém apenas linhas com código IBGE válido
    df = df[df["CODIGO"].notna()].copy()
    df["CODIGO"]     = df["CODIGO"].astype("Int64")
    df["MUNICIPIOS"] = df["MUNICIPIOS"].astype(str)

    return df


def numeric_columns(df: pd.DataFrame, skip: set[str]) -> list[str]:
    """Retorna colunas numéricas do df, ignorando as listadas em skip e a coluna 'total'."""
    cols = []
    for col in df.columns:
        if col in skip:
            continue
        if normalize_text(col) in {"total", ""}:
            continue
        cols.append(col)
    return cols


def sorted_files(directory: Path, pattern: str) -> list[Path]:
    """Retorna arquivos do diretório que casam com pattern, ordenados por ano no nome."""
    files = list(directory.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em: {directory}/{pattern}")
    return sorted(files, key=lambda p: extract_year(p.name))



---
## 5. Mapeamento de Faixas Etárias

Os arquivos brutos usam faixas de 5 em 5 anos (0-4, 5-9, 10-14 …). Esta função
agrupa essas faixas nas 8 categorias padronizadas definidas em `FAIXA_TARGET`,
unificando 70-79 e 80+ em **"70 ou mais"**.

In [20]:
def map_faixa(col: str) -> str | None:
    c = normalize_text(col).replace(" ", "")
    mapa = {
        "00-04": "0-9",  "0-4":  "0-9",  "05-09": "0-9",  "5-9":  "0-9",
        "10-14": "10-19", "15-19": "10-19",
        "20-24": "20-29", "25-29": "20-29",
        "30-34": "30-39", "35-39": "30-39",
        "40-44": "40-49", "45-49": "40-49",
        "50-54": "50-59", "55-59": "50-59",
        "60-64": "60-69", "65-69": "60-69",
        "70-74": "70 ou mais", "75-79": "70 ou mais", "70-79": "70 ou mais",
        "80-84": "70 ou mais", "85+":   "70 ou mais", "85-89": "70 ou mais",
        "90+":   "70 ou mais", "80oumais": "70 ou mais", "80+": "70 ou mais",
    }
    return mapa.get(c)


---
## 6. Funções de Agregação

### 6.1 Ano de Diagnóstico
Agrega todos os arquivos anuais em uma única tabela com uma coluna por ano (1985–2023),
somando os casos por município.

In [21]:
def aggregate_ano_diagnostico(raw_files: list[Path]) -> pd.DataFrame:
    """
    Lê todos os arquivos de Ano de Diagnóstico e os concatena em uma única tabela.
    Cada coluna representa um ano (1985–2023); linhas = municípios.
    """
    frames: list[pd.DataFrame] = []
    for file_path in raw_files:
        df   = read_raw_table(file_path)
        cols = numeric_columns(df, {df.columns[0], "MUNICIPIOS", "CODIGO"})
        rename_map = {}
        for col in cols:
            norm = clean_col(col)
            rename_map[col] = "Sem Informacao" if normalize_text(norm) == "sem informacao" else norm
        current = df[["MUNICIPIOS", "CODIGO", *cols]].rename(columns=rename_map)
        frames.append(current)

    long_df    = pd.concat(frames, ignore_index=True)
    value_cols = [c for c in long_df.columns if c not in {"MUNICIPIOS", "CODIGO"}]
    for col in value_cols:
        long_df[col] = pd.to_numeric(long_df[col], errors="coerce")

    agg = long_df.groupby(["MUNICIPIOS", "CODIGO"], as_index=False)[value_cols].sum(min_count=1)

    # Ordena colunas: anos primeiro, depois demais
    ano_cols  = sorted(
        [c for c in value_cols if re.fullmatch(r"\d{4}", str(c)) and ANO_MIN_VALIDO <= int(c) <= ANO_MAX_VALIDO],
        key=int,
    )
    tail_cols = sorted([c for c in value_cols if c not in ano_cols and not re.fullmatch(r"\d{4}", str(c))])
    ordered   = ["MUNICIPIOS", "CODIGO", *ano_cols, *tail_cols]

    return agg[ordered].rename(columns={"MUNICIPIOS": "MUNICÍPIOS", "CODIGO": "CÓDIGO"})


### 6.2 Dimensões categóricas (Escolaridade, Raça/Cor e Faixa Etária)

Agrega os arquivos anuais em formato **wide**: cada coluna recebe o nome `YYYY_categoria`.
Para Faixa Etária, as colunas de 5 em 5 anos são agrupadas nas 8 faixas padronizadas.

In [22]:
def aggregate_wide_by_year(
    raw_files: list[Path],
    category_mapper,
    *,
    faixa_mode: bool = False,
) -> pd.DataFrame:
    """
    Itera sobre os arquivos brutos (um por ano) e monta uma tabela wide com colunas
    no formato YYYY_categoria.

    Parâmetros
    ----------
    raw_files       : lista de arquivos brutos ordenados por ano
    category_mapper : função que recebe o nome bruto da coluna e retorna o nome padronizado
    faixa_mode      : se True, usa map_faixa() para agrupar faixas de 5 em 5 anos nas 8 target
    """
    result: pd.DataFrame | None = None

    for file_path in sorted(raw_files, key=lambda p: extract_year(p.name)):
        year = extract_year(file_path.name)
        df   = read_raw_table(file_path)
        cols = numeric_columns(df, {df.columns[0], "MUNICIPIOS", "CODIGO"})

        work = df[["MUNICIPIOS", "CODIGO", *cols]].copy()
        for col in cols:
            work[col] = pd.to_numeric(work[col], errors="coerce")

        if faixa_mode:
            # Agrupa colunas brutas nas 8 faixas padronizadas
            grouped_cols: dict[str, list[str]] = {k: [] for k in FAIXA_TARGET}
            for col in cols:
                mapped = map_faixa(col)
                if mapped is not None:
                    grouped_cols[mapped].append(col)

            current = work[["MUNICIPIOS", "CODIGO"]].copy()
            for target in FAIXA_TARGET:
                src = grouped_cols[target]
                current[f"{year}_{target}"] = work[src].sum(axis=1, min_count=1) if src else pd.NA
        else:
            renamed = {col: f"{year}_{category_mapper(col)}" for col in cols}
            current = work[["MUNICIPIOS", "CODIGO", *cols]].rename(columns=renamed)

        result = current if result is None else result.merge(current, on=["MUNICIPIOS", "CODIGO"], how="outer")

    if result is None:
        raise ValueError("Nenhum arquivo processado.")

    return result.rename(columns={"MUNICIPIOS": "MUNICÍPIOS", "CODIGO": "CÓDIGO"})


def normalize_categoria(col: str) -> str:
    """Padroniza nome de categoria: retorna 'Sem Informacao' para variações de 'sem informação',
    caso contrário retorna o nome limpo pelo clean_col."""
    normalized = clean_col(col)
    return "Sem Informacao" if normalize_text(normalized) == "sem informacao" else normalized


---
## 7. Pipeline de Tratamento

Executa o pipeline completo em 5 etapas:
1. Localiza os arquivos brutos de cada dimensão
2. Agrega Ano de Diagnóstico
3. Agrega Escolaridade, Faixa Etária e Raça/Cor (formato wide)
4. Une todas as tabelas em uma tabela consolidada
5. Salva os CSVs de saída

In [23]:
# ── Etapa 1: Localizar arquivos ───────────────────────────────────────────────
print("Arquivos brutos:")
files_ano    = sorted_files(DIR_ANO_DIAG,     "AnoDiagnostico-*.csv")
files_esc    = sorted_files(DIR_ESCOLARIDADE,  "Escolaridade-*.csv")
files_faixa  = sorted_files(DIR_FAIXA_ETARIA,  "Faixa-Etaria-*.csv")
files_raca   = sorted_files(DIR_RACA_COR,      "RacaCor-*.csv")

print(f"   Ano Diagnóstico : {len(files_ano)} arquivo(s)")
print(f"   Escolaridade    : {len(files_esc)} arquivo(s)")
print(f"   Faixa Etária    : {len(files_faixa)} arquivo(s)")
print(f"   Raça/Cor        : {len(files_raca)} arquivo(s)")

Arquivos brutos:
   Ano Diagnóstico : 38 arquivo(s)
   Escolaridade    : 38 arquivo(s)
   Faixa Etária    : 38 arquivo(s)
   Raça/Cor        : 38 arquivo(s)


In [24]:
# ── Etapa 2: Ano de Diagnóstico ──────────────────────────────────────────────
print("\nProcessando Ano de Diagnóstico...")
df_ano_diag = aggregate_ano_diagnostico(files_ano)
print(f"   Shape: {df_ano_diag.shape} | Municípios: {df_ano_diag['MUNICÍPIOS'].nunique()}")
df_ano_diag.head(3)


Processando Ano de Diagnóstico...
   Shape: (853, 51) | Municípios: 853


,MUNICÍPIOS,CÓDIGO,1985,1986,1987,1988,1989,1990,1991,1992,...,Sem Informacao,Unnamed: 19,Unnamed: 23,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 31,Unnamed: 32
0,abadia dos dourados,3100104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,abaete,3100203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,abre campo,3100302,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# ── Etapa 3a: Escolaridade ───────────────────────────────────────────────────
print("Processando Escolaridade...")
df_escolaridade = aggregate_wide_by_year(files_esc, normalize_categoria)
print(f"   Shape: {df_escolaridade.shape} | Municípios: {df_escolaridade['MUNICÍPIOS'].nunique()}")

Processando Escolaridade...
   Shape: (853, 163) | Municípios: 853


In [26]:
# ── Etapa 3b: Faixa Etária ───────────────────────────────────────────────────
print("Processando Faixa Etária...")
df_faixa_etaria = aggregate_wide_by_year(files_faixa, lambda c: c, faixa_mode=True)
print(f"   Shape: {df_faixa_etaria.shape} | Municípios: {df_faixa_etaria['MUNICÍPIOS'].nunique()}")

Processando Faixa Etária...
   Shape: (853, 306) | Municípios: 853


In [27]:
# ── Etapa 3c: Raça/Cor ───────────────────────────────────────────────────────
print("Processando Raça/Cor...")
df_raca_cor = aggregate_wide_by_year(files_raca, normalize_categoria)
print(f"   Shape: {df_raca_cor.shape} | Municípios: {df_raca_cor['MUNICÍPIOS'].nunique()}")

Processando Raça/Cor...
   Shape: (853, 150) | Municípios: 853


In [28]:
# ── Etapa 4: União das tabelas ────────────────────────────────────────────────
print("\nUnindo todas as tabelas...")
tabela_unida = (
    df_ano_diag
    .merge(df_escolaridade, on=["MUNICÍPIOS", "CÓDIGO"], how="outer")
    .merge(df_faixa_etaria, on=["MUNICÍPIOS", "CÓDIGO"], how="outer")
    .merge(df_raca_cor,     on=["MUNICÍPIOS", "CÓDIGO"], how="outer")
)
print(f"   Shape tabela unida: {tabela_unida.shape}")


Unindo todas as tabelas...
   Shape tabela unida: (853, 664)


In [29]:
# ── Etapa 5: Salvar CSVs ─────────────────────────────────────────────────────
print("\n💾 Salvando arquivos...")

df_ano_diag.to_csv(SAIDA_ANO_DIAG,     index=False)
df_escolaridade.to_csv(SAIDA_ESCOLARIDADE, index=False)
df_faixa_etaria.to_csv(SAIDA_FAIXA_ETARIA, index=False)
df_raca_cor.to_csv(SAIDA_RACA_COR,     index=False)
tabela_unida.to_csv(SAIDA_TABELA_UNIDA, index=False)



💾 Salvando arquivos...


---
## 8. Inspeção Pós-Processamento

Verificação rápida dos arquivos gerados: shape, primeiras linhas e contagem de valores ausentes.

In [30]:
from IPython.display import display, Markdown

tabelas = {
    "Ano de Diagnóstico": df_ano_diag,
    "Escolaridade"      : df_escolaridade,
    "Faixa Etária"      : df_faixa_etaria,
    "Raça/Cor"          : df_raca_cor,
    "Tabela Unida"      : tabela_unida,
}

for nome, df in tabelas.items():
    nulos = int(df.isnull().sum().sum())
    display(Markdown(f"**{nome}** — {df.shape[0]} linhas × {df.shape[1]} colunas | nulos totais: {nulos}"))
    display(df.head(2))

**Ano de Diagnóstico** — 853 linhas × 51 colunas | nulos totais: 1837

,MUNICÍPIOS,CÓDIGO,1985,1986,1987,1988,1989,1990,1991,1992,...,Sem Informacao,Unnamed: 19,Unnamed: 23,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 31,Unnamed: 32
0,abadia dos dourados,3100104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,abaete,3100203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


**Escolaridade** — 853 linhas × 163 colunas | nulos totais: 46454

,MUNICÍPIOS,CÓDIGO,1998_Sem Informacao,1999_Nivel medio,1999_Sem Informacao,2000_Fundamental completo,2000_Fundamental incompleto,2000_Nenhuma,2000_Nivel medio,2000_Nivel superior completo,...,2022_Nivel superior completo,2022_Nivel superior incompleto,2022_Sem Informacao,2023_Fundamental completo,2023_Fundamental incompleto,2023_Nenhuma,2023_Nivel medio,2023_Nivel superior completo,2023_Nivel superior incompleto,2023_Sem Informacao
0,abadia dos dourados,3100104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,abaete,3100203,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,8.0,2.0,9.0,0.0,0.0,1.0,0.0,2.0


**Faixa Etária** — 853 linhas × 306 colunas | nulos totais: 157159

,MUNICÍPIOS,CÓDIGO,1985_0-9,1985_10-19,1985_20-29,1985_30-39,1985_40-49,1985_50-59,1985_60-69,1985_70 ou mais,...,2022_60-69,2022_70 ou mais,2023_0-9,2023_10-19,2023_20-29,2023_30-39,2023_40-49,2023_50-59,2023_60-69,2023_70 ou mais
0,abadia dos dourados,3100104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,abaete,3100203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,4.0,0.0,0.0,1.0,1.0,2.0,4.0,4.0,2.0


**Raça/Cor** — 853 linhas × 150 colunas | nulos totais: 45481

,MUNICÍPIOS,CÓDIGO,1998_Branca,1998_Parda,1998_Preta,1998_Sem Informacao,1999_Branca,1999_Preta,1999_Sem Informacao,2000_Amarela,...,2022_Indigena,2022_Parda,2022_Preta,2022_Sem Informacao,2023_Amarela,2023_Branca,2023_Indigena,2023_Parda,2023_Preta,2023_Sem Informacao
0,abadia dos dourados,3100104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,abaete,3100203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,0.0,11.0,0.0,1.0,0.0,2.0,0.0,11.0,1.0,0.0


**Tabela Unida** — 853 linhas × 664 colunas | nulos totais: 250931

,MUNICÍPIOS,CÓDIGO,1985,1986,1987,1988,1989,1990,1991,1992,...,2022_Indigena,2022_Parda,2022_Preta,2022_Sem Informacao_y,2023_Amarela,2023_Branca,2023_Indigena,2023_Parda,2023_Preta,2023_Sem Informacao_y
0,abadia dos dourados,3100104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,abaete,3100203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,11.0,0.0,1.0,0.0,2.0,0.0,11.0,1.0,0.0
